# Module 3 — SQL Analytics Demo
## Jordan National University

| Demo | Concept | Business Question |
|------|---------|-------------------|
| Demo 1 | `JOIN` | Which instructor teaches each course, and what faculty are they from? |
| Demo 2 | `GROUP BY` + `HAVING` | Which courses have a class average above 80? |
| Demo 3 | Window — `RANK` | Which courses have the highest enrollment per faculty? |
| Demo 4 | `LEFT JOIN` + `COALESCE` | Are there any courses with zero enrollments this semester? |
| Demo 5 | CTE | Which faculties employ more instructors than the university average? |


---
## Setup


In [1]:
%pip install ipython-sql prettytable==0.7.2 --quiet
print("✅ ipython-sql + prettytable installed")

Note: you may need to restart the kernel to use updated packages.
✅ ipython-sql + prettytable installed


In [2]:
%load_ext sql
%sql sqlite:///:memory:
print("✅ In-memory SQLite database connected")


/usr/lib/python3/dist-packages/pytz/__init__.py:31: SyntaxWarning: invalid escape sequence '\s'
  match = re.match("^#\s*version\s*([0-9a-z]*)\s*$", line)


✅ In-memory SQLite database connected


In [3]:
%%sql
-- ============================================================
-- SCHEMA: Jordan National University
-- ============================================================
CREATE TABLE IF NOT EXISTS faculties (
    faculty_id  INTEGER PRIMARY KEY AUTOINCREMENT,
    name        TEXT NOT NULL,
    dean        TEXT,
    building    TEXT
);

CREATE TABLE IF NOT EXISTS instructors (
    instructor_id INTEGER PRIMARY KEY AUTOINCREMENT,
    name          TEXT NOT NULL,
    faculty_id    INTEGER NOT NULL REFERENCES faculties(faculty_id),
    rank          TEXT,
    hire_date     TEXT  -- stored as ISO-8601 datetime: 'YYYY-MM-DD HH:MM:SS'
);

CREATE TABLE IF NOT EXISTS courses (
    course_id     INTEGER PRIMARY KEY AUTOINCREMENT,
    course_code   TEXT NOT NULL UNIQUE,  -- e.g. ENG-001, CS-002
    title         TEXT NOT NULL,
    faculty_id    INTEGER NOT NULL REFERENCES faculties(faculty_id),
    instructor_id INTEGER REFERENCES instructors(instructor_id),
    credits       INTEGER DEFAULT 3,
    semester      TEXT
);

CREATE TABLE IF NOT EXISTS enrollments (
    enrollment_id INTEGER PRIMARY KEY AUTOINCREMENT,
    course_id     INTEGER NOT NULL REFERENCES courses(course_id),
    student_name  TEXT NOT NULL,
    grade         REAL,
    enrolled_date TEXT NOT NULL  -- stored as ISO-8601 datetime: 'YYYY-MM-DD HH:MM:SS'
);


 * sqlite:///:memory:
Done.
Done.
Done.
Done.


[]

In [4]:
%%sql
-- ============================================================
-- SEED: faculties (5 rows)
-- ============================================================
INSERT INTO faculties (name, dean, building) VALUES
    ('Faculty of Engineering',       'Dr. Riad Haddad',   'Block A'),
    ('Faculty of Computer Science',  'Dr. Layla Mansour', 'Block B'),
    ('Faculty of Business',          'Dr. Samir Khalil',  'Block C'),
    ('Faculty of Arts & Humanities', 'Dr. Hana Nimri',    'Block D'),
    ('Faculty of Medicine',          'Dr. Ziad Barakat',  'Block E');

-- ============================================================
-- SEED: instructors (12 rows)  — hire_date as full datetime
-- ============================================================
INSERT INTO instructors (name, faculty_id, rank, hire_date) VALUES
('Dr. Omar Saleh',    1, 'Professor',          '2010-09-01 08:00:00'),
('Dr. Rania Khoury',  1, 'Associate Professor','2015-02-15 08:00:00'),
('Dr. Tarek Fahed',   1, 'Assistant Professor','2019-08-20 08:00:00'),
('Dr. Mona Issa',     2, 'Professor',          '2008-09-01 08:00:00'),
('Dr. Ali Hamdan',    2, 'Associate Professor','2013-01-10 08:00:00'),
('Dr. Sara Jabri',    2, 'Assistant Professor','2020-03-01 08:00:00'),
('Dr. Nadia Obeid',   3, 'Professor',          '2009-09-01 08:00:00'),
('Dr. Wael Masri',    3, 'Associate Professor','2016-06-15 08:00:00'),
('Dr. Lara Bitar',    4, 'Professor',          '2011-09-01 08:00:00'),
('Dr. Faris Shahin',  4, 'Assistant Professor','2021-01-20 08:00:00'),
('Dr. Rana Zawaideh', 5, 'Professor',          '2007-09-01 08:00:00'),
('Dr. Heba Srour',    5, 'Associate Professor','2014-05-01 08:00:00');

-- ============================================================
-- SEED: courses (13 rows — course 13 has ZERO enrollments)
-- course_code format: <FACULTY_PREFIX>-<AUTOINCREMENT_ID padded to 3>
-- ============================================================
INSERT INTO courses (course_code, title, faculty_id, instructor_id, credits, semester) VALUES
('ENG-001', 'Structural Engineering I',     1,  1, 3, '2023-Fall'),
('ENG-002', 'Thermodynamics',               1,  2, 3, '2023-Fall'),
('ENG-003', 'Circuit Design',               1,  3, 3, '2023-Fall'),
('CS-001',  'Algorithms & Data Structures', 2,  4, 3, '2023-Fall'),
('CS-002',  'Database Systems',             2,  5, 3, '2023-Fall'),
('CS-003',  'Machine Learning Fundamentals',2,  6, 3, '2023-Fall'),
('BUS-001', 'Business Strategy',            3,  7, 3, '2023-Fall'),
('BUS-002', 'Financial Accounting',         3,  8, 3, '2023-Fall'),
('ART-001', 'Arabic Literature',            4,  9, 3, '2023-Fall'),
('ART-002', 'Modern History',               4, 10, 3, '2023-Fall'),
('MED-001', 'Human Anatomy',                5, 11, 4, '2023-Fall'),
('MED-002', 'Pharmacology',                 5, 12, 4, '2023-Fall'),
('ENG-004', 'Advanced Robotics',            1,  1, 3, '2023-Fall');  -- zero enrollments

-- ============================================================
-- SEED: enrollments (48 rows)  — enrolled_date as full datetime
-- ============================================================
INSERT INTO enrollments (course_id, student_name, grade, enrolled_date) VALUES
-- Course 1: Structural Engineering I  (avg ~75)
(1,'Ahmad Zreiqat', 72.0,'2023-09-01 09:00:00'),(1,'Dina Rawashdeh',80.0,'2023-09-01 09:05:00'),
(1,'Yara Twal',     68.0,'2023-09-01 09:10:00'),(1,'Murad Aqrabawi',76.0,'2023-09-01 09:15:00'),
(1,'Reem Khaled',   79.0,'2023-09-01 09:20:00'),
-- Course 2: Thermodynamics  (avg ~83)
(2,'Loay Salti',    85.0,'2023-09-01 09:00:00'),(2,'Hana Balbisi',  82.0,'2023-09-01 09:05:00'),
(2,'Firas Hijazi',  81.0,'2023-09-01 09:10:00'),(2,'Nour Atiyeh',   84.0,'2023-09-01 09:15:00'),
(2,'Lina Jado',     83.0,'2023-09-01 09:20:00'),
-- Course 3: Circuit Design  (avg ~71)
(3,'Basel Shatnawi',70.0,'2023-09-01 09:00:00'),(3,'Rawan Zawahreh',73.0,'2023-09-01 09:05:00'),
(3,'Salam Bisharat',69.0,'2023-09-01 09:10:00'),(3,'Nura Haj',      72.0,'2023-09-01 09:15:00'),
-- Course 4: Algorithms & Data Structures  (avg ~88)
(4,'Omar Barakat',  90.0,'2023-09-01 09:00:00'),(4,'Lina Nasser',   87.0,'2023-09-01 09:05:00'),
(4,'Sara Mansour',  88.0,'2023-09-01 09:10:00'),(4,'Tarek Haddad',  86.0,'2023-09-01 09:15:00'),
(4,'Mona Fahed',    89.0,'2023-09-01 09:20:00'),(4,'Ali Khalil',    88.0,'2023-09-01 09:25:00'),
-- Course 5: Database Systems  (avg ~79)
(5,'Dalia Rashid',  78.0,'2023-09-01 09:00:00'),(5,'Kareem Saad',   81.0,'2023-09-01 09:05:00'),
(5,'Heba Qasem',    77.0,'2023-09-01 09:10:00'),(5,'Yousef Saleh',  80.0,'2023-09-01 09:15:00'),
-- Course 6: Machine Learning Fundamentals  (avg ~84)
(6,'Rania Hamdan',  85.0,'2023-09-01 09:00:00'),(6,'Basem Ayyad',   83.0,'2023-09-01 09:05:00'),
(6,'Rana Nimri',    84.0,'2023-09-01 09:10:00'),(6,'Ziad Khoury',   84.0,'2023-09-01 09:15:00'),
(6,'Lara Issa',     84.0,'2023-09-01 09:20:00'),
-- Course 7: Business Strategy  (avg ~76)
(7,'Maya Jabri',    75.0,'2023-09-01 09:00:00'),(7,'Sami Awad',     78.0,'2023-09-01 09:05:00'),
(7,'Heba Houri',    74.0,'2023-09-01 09:10:00'),(7,'Jad Dahlan',    77.0,'2023-09-01 09:15:00'),
-- Course 8: Financial Accounting  (avg ~81)
(8,'Lama Kanaan',   82.0,'2023-09-01 09:00:00'),(8,'Ahmad Nassar',  80.0,'2023-09-01 09:05:00'),
(8,'Rami Dabbas',   83.0,'2023-09-01 09:10:00'),(8,'Lena Farhat',   79.0,'2023-09-01 09:15:00'),
-- Course 9: Arabic Literature  (avg ~72)
(9,'Khalid Madi',   72.0,'2023-09-01 09:00:00'),(9,'Sana Jaber',    74.0,'2023-09-01 09:05:00'),
(9,'Nidal Zreiqat', 71.0,'2023-09-01 09:10:00'),
-- Course 10: Modern History  (avg ~82)
(10,'Hana Asmar',   83.0,'2023-09-01 09:00:00'),(10,'Wael Hadid',   81.0,'2023-09-01 09:05:00'),
(10,'Deema Halabi', 82.0,'2023-09-01 09:10:00'),(10,'Nadia Farraj', 82.0,'2023-09-01 09:15:00'),
-- Course 11: Human Anatomy  (avg ~86)
(11,'Eyad Sabbah',  87.0,'2023-09-01 09:00:00'),(11,'Roula Qudah',  85.0,'2023-09-01 09:05:00'),
(11,'Waseem Khalaf',86.0,'2023-09-01 09:10:00'),(11,'Abeer Salti',  86.0,'2023-09-01 09:15:00'),
(11,'Dina Hijazi',  86.0,'2023-09-01 09:20:00'),
-- Course 12: Pharmacology  (avg ~77)
(12,'Tariq Obeid',  78.0,'2023-09-01 09:00:00'),(12,'Yara Bitar',   76.0,'2023-09-01 09:05:00'),
(12,'Firas Masri',  77.0,'2023-09-01 09:10:00'),(12,'Nour Shahin',  77.0,'2023-09-01 09:15:00');
-- Course 13 (Advanced Robotics / ENG-004): ZERO enrollments — intentional for Demo 4


 * sqlite:///:memory:
5 rows affected.
12 rows affected.
13 rows affected.
53 rows affected.
Done.


[]

In [5]:
%%sql
SELECT 'faculties'   AS tbl, COUNT(*) AS rows FROM faculties   UNION ALL
SELECT 'instructors',         COUNT(*)         FROM instructors UNION ALL
SELECT 'courses',             COUNT(*)         FROM courses     UNION ALL
SELECT 'enrollments',         COUNT(*)         FROM enrollments;
-- Expected: 5 / 12 / 13 / 48


 * sqlite:///:memory:
Done.
Done.


[]

---
## 📐 Schema Reference

```
faculties                        instructors
─────────────────────            ──────────────────────────────────────
faculty_id  INTEGER  PK  ◄────  faculty_id  INTEGER  FK
name        TEXT               │ instructor_id INTEGER PK
dean        TEXT               │ name        TEXT
building    TEXT               │ rank        TEXT
                               │ hire_date   TEXT (ISO datetime)
                               │
courses                        │ enrollments
──────────────────────         │ ──────────────────────────────
course_id     INTEGER PK  ◄───┼─ course_id     INTEGER FK
course_code   TEXT UNIQUE     │  enrollment_id INTEGER PK
title         TEXT            │  student_name   TEXT
faculty_id    INTEGER FK ─────┘  grade          REAL (nullable)
instructor_id INTEGER FK ──────► instructors.instructor_id
credits       INTEGER            enrolled_date  TEXT (ISO datetime)
semester      TEXT
```

- **course_code** format: `<FACULTY_PREFIX>-<3-digit sequence>` (e.g. `ENG-001`, `CS-002`)
- **hire_date / enrolled_date** stored as ISO-8601 datetime strings (`'YYYY-MM-DD HH:MM:SS'`)
- `Advanced Robotics` (`ENG-004`) has **zero enrollments** — intentional for Demo 4
- Engineering has **3 instructors**, all other faculties **2** — intentional for Demo 5


---
## Demo 1 — JOIN
> *"Which instructor teaches each course, and what faculty are they from?"*


In [6]:
%%sql
-- Step 1: explore instructors — notice faculty_id is just a number
SELECT * FROM instructors LIMIT 5;


 * sqlite:///:memory:
Done.


instructor_id,name,faculty_id,rank,hire_date
1,Dr. Omar Saleh,1,Professor,2010-09-01 08:00:00
2,Dr. Rania Khoury,1,Associate Professor,2015-02-15 08:00:00
3,Dr. Tarek Fahed,1,Assistant Professor,2019-08-20 08:00:00
4,Dr. Mona Issa,2,Professor,2008-09-01 08:00:00
5,Dr. Ali Hamdan,2,Associate Professor,2013-01-10 08:00:00


In [7]:
%%sql
-- Step 2: explore faculties — this has the human-readable name
SELECT * FROM faculties;


 * sqlite:///:memory:
Done.


faculty_id,name,dean,building
1,Faculty of Engineering,Dr. Riad Haddad,Block A
2,Faculty of Computer Science,Dr. Layla Mansour,Block B
3,Faculty of Business,Dr. Samir Khalil,Block C
4,Faculty of Arts & Humanities,Dr. Hana Nimri,Block D
5,Faculty of Medicine,Dr. Ziad Barakat,Block E


In [8]:
%%sql
-- Step 3: JOIN on the shared key (faculty_id) to combine both tables
-- Also show course_code so students see it alongside the instructor
SELECT
    c.course_code,
    c.title         AS course_title,
    i.name          AS instructor_name,
    i.rank,
    f.name          AS faculty_name
FROM courses c
JOIN instructors i ON c.instructor_id = i.instructor_id
JOIN faculties   f ON c.faculty_id    = f.faculty_id
ORDER BY f.name, c.course_code;


 * sqlite:///:memory:
Done.


course_code,course_title,instructor_name,rank,faculty_name
ART-001,Arabic Literature,Dr. Lara Bitar,Professor,Faculty of Arts & Humanities
ART-002,Modern History,Dr. Faris Shahin,Assistant Professor,Faculty of Arts & Humanities
BUS-001,Business Strategy,Dr. Nadia Obeid,Professor,Faculty of Business
BUS-002,Financial Accounting,Dr. Wael Masri,Associate Professor,Faculty of Business
CS-001,Algorithms & Data Structures,Dr. Mona Issa,Professor,Faculty of Computer Science
CS-002,Database Systems,Dr. Ali Hamdan,Associate Professor,Faculty of Computer Science
CS-003,Machine Learning Fundamentals,Dr. Sara Jabri,Assistant Professor,Faculty of Computer Science
ENG-001,Structural Engineering I,Dr. Omar Saleh,Professor,Faculty of Engineering
ENG-002,Thermodynamics,Dr. Rania Khoury,Associate Professor,Faculty of Engineering
ENG-003,Circuit Design,Dr. Tarek Fahed,Assistant Professor,Faculty of Engineering


| Clause | What it does |
|--------|--------------|
| `FROM courses c` | Start here — each row is one course. |
| `JOIN instructors i ON c.instructor_id = i.instructor_id` | Attach the instructor for each course. |
| `JOIN faculties f ON c.faculty_id = f.faculty_id` | Attach the faculty name. |
| `c.course_code` | Human-readable code that encodes both sequence and faculty. |

**Key insight:** `JOIN` only returns rows with a match in both tables. Every course here has an instructor and a faculty, so all 13 appear.


---
## Demo 2 — GROUP BY + HAVING
> *"Which courses have a strong class performance — average grade above 80?"*


In [9]:
%%sql
-- Step 1: raw average per course (course_id only — no labels yet)
SELECT course_id, AVG(grade) AS avg_grade
FROM enrollments
GROUP BY course_id;


 * sqlite:///:memory:
Done.


course_id,avg_grade
1,75.0
2,83.0
3,71.0
4,88.0
5,79.0
6,84.0
7,76.0
8,81.0
9,72.33333333333333
10,82.0


In [10]:
%%sql
-- Step 2: join course title + filter with HAVING
-- HAVING filters AFTER grouping; WHERE cannot see AVG() results
SELECT
    c.course_code,
    c.title                  AS course_title,
    ROUND(AVG(e.grade), 2)   AS avg_grade,
    COUNT(e.enrollment_id)   AS student_count
FROM enrollments e
JOIN courses c ON e.course_id = c.course_id
GROUP BY c.course_id, c.course_code, c.title
HAVING AVG(e.grade) > 80
ORDER BY avg_grade DESC;


 * sqlite:///:memory:
Done.


course_code,course_title,avg_grade,student_count
CS-001,Algorithms & Data Structures,88.0,6
MED-001,Human Anatomy,86.0,5
CS-003,Machine Learning Fundamentals,84.0,5
ENG-002,Thermodynamics,83.0,5
ART-002,Modern History,82.0,4
BUS-002,Financial Accounting,81.0,4


| Clause | What it does |
|--------|--------------|
| `GROUP BY c.course_id, c.course_code, c.title` | One row per course — all non-aggregate SELECT columns must be listed. |
| `HAVING AVG(e.grade) > 80` | Filters **after** aggregation. Use `HAVING` for aggregate conditions, `WHERE` for row-level conditions. |
| `ORDER BY avg_grade DESC` | Highest average first. |


---
## Demo 3 — Window Function: RANK
> *"For each faculty, which courses have the highest enrollment? Rank them."*


In [11]:
%%sql
-- Step 1: enrollment count per course (inner query)
SELECT
    f.name                 AS faculty_name,
    c.course_code,
    c.title                AS course_title,
    COUNT(e.enrollment_id) AS enrollment_count
FROM courses c
JOIN faculties f ON c.faculty_id = f.faculty_id
LEFT JOIN enrollments e ON c.course_id = e.course_id
GROUP BY f.name, c.course_code, c.title
ORDER BY f.name, COUNT(e.enrollment_id) DESC;


 * sqlite:///:memory:
Done.


faculty_name,course_code,course_title,enrollment_count
Faculty of Arts & Humanities,ART-002,Modern History,4
Faculty of Arts & Humanities,ART-001,Arabic Literature,3
Faculty of Business,BUS-001,Business Strategy,4
Faculty of Business,BUS-002,Financial Accounting,4
Faculty of Computer Science,CS-001,Algorithms & Data Structures,6
Faculty of Computer Science,CS-003,Machine Learning Fundamentals,5
Faculty of Computer Science,CS-002,Database Systems,4
Faculty of Engineering,ENG-001,Structural Engineering I,5
Faculty of Engineering,ENG-002,Thermodynamics,5
Faculty of Engineering,ENG-003,Circuit Design,4


In [12]:
%%sql
-- Step 2: wrap in subquery and add RANK() partitioned by faculty
-- RANK() ties share the same rank number; the next rank skips accordingly
SELECT
    faculty_name,
    course_code,
    course_title,
    enrollment_count,
    RANK() OVER (
        PARTITION BY faculty_name
        ORDER BY enrollment_count DESC
    ) AS enrollment_rank
FROM (
    SELECT
        f.name                 AS faculty_name,
        c.course_code,
        c.title                AS course_title,
        COUNT(e.enrollment_id) AS enrollment_count
    FROM courses c
    JOIN faculties f ON c.faculty_id = f.faculty_id
    LEFT JOIN enrollments e ON c.course_id = e.course_id
    GROUP BY f.name, c.course_code, c.title
) sub
ORDER BY faculty_name, enrollment_rank;


 * sqlite:///:memory:
Done.


faculty_name,course_code,course_title,enrollment_count,enrollment_rank
Faculty of Arts & Humanities,ART-002,Modern History,4,1
Faculty of Arts & Humanities,ART-001,Arabic Literature,3,2
Faculty of Business,BUS-001,Business Strategy,4,1
Faculty of Business,BUS-002,Financial Accounting,4,1
Faculty of Computer Science,CS-001,Algorithms & Data Structures,6,1
Faculty of Computer Science,CS-003,Machine Learning Fundamentals,5,2
Faculty of Computer Science,CS-002,Database Systems,4,3
Faculty of Engineering,ENG-001,Structural Engineering I,5,1
Faculty of Engineering,ENG-002,Thermodynamics,5,1
Faculty of Engineering,ENG-003,Circuit Design,4,3


| Clause | What it does |
|--------|--------------|
| `RANK() OVER (...)` | Computes a rank per row without collapsing rows (unlike `GROUP BY`). |
| `PARTITION BY faculty_name` | Restarts ranking at 1 for each faculty independently. |
| `ORDER BY enrollment_count DESC` | Highest enrollment = rank 1 within each partition. |
| Ties | `RANK()` assigns the same number to ties and skips the next rank. Use `ROW_NUMBER()` for strictly unique ranks. |


---
## Demo 4 — LEFT JOIN + COALESCE
> *"Are there any courses that no student has enrolled in this semester?"*


In [13]:
%%sql
-- Step 1: INNER JOIN — intentionally broken; shows only 12 courses (ENG-004 disappears)
SELECT
    c.course_code,
    c.title,
    COUNT(e.enrollment_id) AS enroll_count
FROM courses c
JOIN enrollments e ON c.course_id = e.course_id
GROUP BY c.course_code, c.title
ORDER BY enroll_count;


 * sqlite:///:memory:
Done.


course_code,title,enroll_count
ART-001,Arabic Literature,3
ART-002,Modern History,4
BUS-001,Business Strategy,4
BUS-002,Financial Accounting,4
CS-002,Database Systems,4
ENG-003,Circuit Design,4
MED-002,Pharmacology,4
CS-003,Machine Learning Fundamentals,5
ENG-001,Structural Engineering I,5
ENG-002,Thermodynamics,5


In [14]:
%%sql
-- Step 2: LEFT JOIN — keeps ALL courses; missing enrollments become 0
-- COUNT(e.enrollment_id) skips NULLs naturally, returning 0 for no-match rows
SELECT
    c.course_code,
    c.title                             AS course_title,
    f.name                              AS faculty_name,
    COALESCE(COUNT(e.enrollment_id), 0) AS enrollment_count
FROM courses c
JOIN faculties f ON c.faculty_id = f.faculty_id
LEFT JOIN enrollments e ON c.course_id = e.course_id
GROUP BY c.course_id, c.course_code, c.title, f.name
ORDER BY enrollment_count ASC, c.course_code;


 * sqlite:///:memory:
Done.


course_code,course_title,faculty_name,enrollment_count
ENG-004,Advanced Robotics,Faculty of Engineering,0
ART-001,Arabic Literature,Faculty of Arts & Humanities,3
ART-002,Modern History,Faculty of Arts & Humanities,4
BUS-001,Business Strategy,Faculty of Business,4
BUS-002,Financial Accounting,Faculty of Business,4
CS-002,Database Systems,Faculty of Computer Science,4
ENG-003,Circuit Design,Faculty of Engineering,4
MED-002,Pharmacology,Faculty of Medicine,4
CS-003,Machine Learning Fundamentals,Faculty of Computer Science,5
ENG-001,Structural Engineering I,Faculty of Engineering,5


| Clause | What it does |
|--------|--------------|
| `FROM courses c` | Start with **all** courses — the "left" table. |
| `LEFT JOIN enrollments e ...` | Keep every course row; no-match rows get `NULL` for all `e.*` columns. |
| `COUNT(e.enrollment_id)` | Skips NULLs — returns 0 when no enrollment match exists. |
| `COALESCE(..., 0)` | Belt-and-suspenders: replaces any remaining NULL with 0. |

**`ENG-004` Advanced Robotics** appears at the top with 0 enrollments — the exact data gap we needed to find.


---
## Demo 5 — CTE
> *"Which faculties employ more instructors than the university-wide average?"*


In [15]:
%%sql
-- Step 1: run the first CTE alone to inspect its output (one row = university avg)
WITH university_avg AS (
    SELECT AVG(instructor_count) AS avg_instructors
    FROM (
        SELECT faculty_id, COUNT(*) AS instructor_count
        FROM instructors
        GROUP BY faculty_id
    ) counts
)
SELECT * FROM university_avg;


 * sqlite:///:memory:
Done.


avg_instructors
2.4


In [16]:
%%sql
-- Step 2: add faculty_counts CTE and compare each faculty against the avg
-- university_avg has one row, so the implicit cross-join just adds that column everywhere
WITH university_avg AS (
    SELECT AVG(instructor_count) AS avg_instructors
    FROM (
        SELECT faculty_id, COUNT(*) AS instructor_count
        FROM instructors
        GROUP BY faculty_id
    ) counts
),
faculty_counts AS (
    SELECT
        f.name                 AS faculty_name,
        COUNT(i.instructor_id) AS instructor_count
    FROM faculties f
    LEFT JOIN instructors i ON f.faculty_id = i.faculty_id
    GROUP BY f.faculty_id, f.name
)
SELECT
    fc.faculty_name,
    fc.instructor_count,
    ROUND(ua.avg_instructors, 2) AS university_avg
FROM faculty_counts fc, university_avg ua
WHERE fc.instructor_count > ua.avg_instructors
ORDER BY fc.instructor_count DESC;


 * sqlite:///:memory:
Done.


faculty_name,instructor_count,university_avg
Faculty of Engineering,3,2.4
Faculty of Computer Science,3,2.4


| Clause | What it does |
|--------|--------------|
| `WITH university_avg AS (...)` | First CTE: compute the university-wide average instructor count. |
| `faculty_counts AS (...)` | Second CTE (separated by a comma): count instructors per faculty. |
| `FROM faculty_counts fc, university_avg ua` | Reference both CTEs like tables. One-row `university_avg` acts as a constant column. |
| `WHERE fc.instructor_count > ua.avg_instructors` | Filter to faculties above the university average. |

**Why CTEs?** Each step is named and readable top-to-bottom. The same logic as nested subqueries — but far easier to follow and debug.


---
## 📋 Concept Summary

| Demo | Concept | When to use it |
|------|---------|---------------|
| 1 | `JOIN` | Combine columns from two or more tables on a shared key |
| 2 | `GROUP BY` + `HAVING` | Aggregate rows into groups; filter on aggregate results |
| 3 | Window — `RANK` | Rank rows **within** partitions without collapsing them |
| 4 | `LEFT JOIN` + `COUNT(col)` | Include rows with no right-side match (zeros, missing data) |
| 5 | CTE | Break a multi-step query into named, readable intermediate results |

### Common errors
- **`WHERE` instead of `HAVING`** on aggregates — `WHERE` runs before aggregation and can't see `AVG()`, `SUM()`, etc.
- **Missing `GROUP BY` columns** — every non-aggregate `SELECT` column must appear in `GROUP BY` (PostgreSQL enforces this strictly).
- **`INNER JOIN` when you need `LEFT JOIN`** — if a course with zero enrollments disappears from your result, you used the wrong join.
- **Window function in `WHERE`** — wrap it in a subquery or CTE first, then filter in the outer query.
